# Creador de Clips Virales - Acertijos para IG

Notebook para **Google Colab**. Genera videos de 30-60 s con voz y subtitulos.

---
## Como usar en Colab

1. **Abrir:** **File > Open notebook > GitHub** > repo `yoquelvisdev08/acertijo` > abre **creador_acertijos_ig.ipynb**.
2. **GPU:** **Runtime > Change runtime type > Hardware accelerator > GPU** (T4 o superior).
3. Ejecuta las celdas en orden. Opcional: **Celda 8b** para pre-descargar modelos antes de la interfaz.
4. TTS en CPU; video en GPU. En la interfaz escribe el acertijo (o un tema corto) y pulsa **Generar**. Descarga el MP4 desde el reproductor.

**Tiempo:** Primera vez 8-15 min (descarga modelos); siguientes ~6-11 min. Revisa la **Consola (logs)** en la interfaz.

**Salida:** Los videos se guardan en `/content/salida_acertijos/`. El enlace publico de Gradio expira en 7 dias (normal en Colab gratis).

In [ ]:
# Colab: verificar entorno. Ejecuta las siguientes celdas en orden.
print('Entorno: Google Colab. Siguiente: instalar MoviePy y dependencias.')

In [1]:
import sys
print('Instalando MoviePy 2.x (evita errores con decorator en Colab)...')
!{sys.executable} -m pip install -q "moviepy>=2.0.0"
print('MoviePy instalado.')

Reinstalando moviepy para asegurar que el modulo este disponible...
Instalacion de moviepy completada.


In [2]:
try:
    from moviepy import VideoFileClip, AudioFileClip, CompositeVideoClip, TextClip
    print('MoviePy 2 listo.')
except Exception as e:
    print(f'Error: {e}. Ejecuta la celda anterior y luego Runtime > Restart runtime.')

Verificando importacion de moviepy.editor...
Error al importar moviepy.editor: No module named 'moviepy.editor'
Por favor, reinicia el entorno de ejecucion (Runtime > Restart runtime) y ejecuta las celdas 1, 2, 3, 4, 5, 6, 7, 8, 8b y luego intenta de nuevo.


In [3]:
# Celda 0 (opcional): Clonar o actualizar repo desde GitHub. Luego abre el .ipynb desde la carpeta clonada.
REPO_URL = 'https://github.com/yoquelvisdev08/acertijo.git'
REPO_DIR = '/content/acertijo'
import os
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
    print('Repo actualizado. Si cambiaste el notebook, reabrelo desde File > Open notebook.')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f'Clonado en {REPO_DIR}. Abre el notebook desde ahi: File > Open notebook > creador_acertijos_ig.ipynb')

hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.
Repo actualizado. Si cambiaste el notebook, reabrelo desde File > Open notebook.


In [4]:
# Celda 1: Instalacion (ejecutar una vez en Colab)
!apt-get update -qq && apt-get install -qq -y imagemagick sox > /dev/null 2>&1
!pip install -q "qwen-tts>=0.1.0" soundfile "diffusers>=0.34.0" "transformers>=4.45.0" accelerate ftfy "moviepy>=2.0.0" pysrt "gradio>=4.0.0"
print('Dependencias instaladas (MoviePy 2.x, qwen-tts, diffusers, gradio).')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencias instaladas.


In [5]:
# Celda 2: Imports y configuracion (Colab)
import os
from pathlib import Path
OUTPUT_DIR = Path('/content/salida_acertijos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
import numpy as np
import soundfile as sf
import re

if not torch.cuda.is_available():
    print('AVISO: No hay GPU. Runtime > Change runtime type > Hardware accelerator > GPU (T4).')
else:
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}. VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

TARGET_VIDEO_DURATION_SEC = (30, 60)
def sanitize_script(text: str, max_len: int = 1500) -> str:
    if not text or not isinstance(text, str):
        return ''
    s = text.strip()
    s = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', s)
    return s[:max_len] if len(s) > max_len else s

def sanitize_video_prompt(text: str, max_len: int = 200) -> str:
    if not text or not isinstance(text, str):
        return 'smooth fluid motion, cinematic, engaging, mysterious, vertical 9:16'
    s = text.strip()
    s = re.sub(r'[^a-zA-Z0-9\s,\.\-]', '', s)
    return (s[:max_len] if len(s) > max_len else s) or 'smooth fluid motion, cinematic, engaging, mysterious, vertical 9:16'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_CPU_FOR_TTS = True
print(f'Dispositivo: {DEVICE}. TTS en CPU. Video en GPU (recomendado T4).')

GPU detectada: Tesla T4. VRAM: 15.6 GB
Dispositivo: cuda. TTS en CPU. Video en GPU (obligatorio en T4 para no saturar RAM).


In [6]:
# Celda 3: Voz con Qwen3-TTS (espanol). Por defecto usa GPU; USE_CPU_FOR_TTS=True en Celda 2 para forzar CPU si hay CUDA assert.
def get_tts_model():
    import time
    print('[TTS] Estamos en: descargando/cargando modelo de VOZ (Qwen3-TTS). Este modelo narra el acertijo.', flush=True)
    t0 = time.time()
    from qwen_tts import Qwen3TTSModel
    tts_device = 'cpu' if USE_CPU_FOR_TTS else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[TTS] Dispositivo: {tts_device} (GPU = menos RAM, mas rapido). Descargando pesos si es la primera vez...', flush=True)
    model = Qwen3TTSModel.from_pretrained(
        'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',
        device_map=tts_device,
        torch_dtype=torch.float16 if tts_device == 'cuda' else torch.float32,
    )
    print(f'[TTS] Modelo de voz listo en {time.time()-t0:.1f}s', flush=True)
    return model

SPEAKERS_VALID = ('Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee', 'Uncle_Fu', 'Dylan', 'Eric')

def generate_voice(text: str, out_path: str, speaker: str = 'Ryan', model=None):
    import time
    t0 = time.time()
    print('[TTS] Iniciando generacion de voz...', flush=True)
    if model is None:
        model = get_tts_model()
    text = sanitize_script(text, max_len=1500)
    if not text:
        raise ValueError('Texto del acertijo vacio tras sanitizar.')
    speaker = speaker if speaker in SPEAKERS_VALID else 'Ryan'
    wavs, sr = model.generate_custom_voice(
        text=text,
        language='Spanish',
        speaker=speaker,
    )
    sf.write(out_path, wavs[0], sr)
    dur = len(wavs[0]) / sr
    print(f'[TTS] Listo en {time.time()-t0:.1f}s (audio {dur:.1f}s)', flush=True)
    return out_path, sr, dur

In [7]:
# Celda 4: Video con ModelScope text-to-video-ms-1.7b (Colab con GPU T4)
def get_video_pipeline():
    import time
    import gc
    if not torch.cuda.is_available():
        raise RuntimeError('Se necesita GPU. Runtime > Change runtime type > GPU (T4).')
    from diffusers import DiffusionPipeline
    from diffusers.utils import export_to_video
    model_id = 'damo-vilab/text-to-video-ms-1.7b'
    print('[VIDEO] Cargando modelo de VIDEO en GPU (ModelScope 1.7B)...', flush=True)
    t0 = time.time()
    gc.collect()
    torch.cuda.empty_cache()
    pipeline = DiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        variant='fp16',
    )
    pipeline = pipeline.to('cuda')
    if hasattr(pipeline.vae, 'enable_slicing'):
        pipeline.vae.enable_slicing()
    elif hasattr(pipeline, 'enable_vae_slicing'):
        pipeline.enable_vae_slicing()
    gc.collect()
    torch.cuda.empty_cache()
    print(f'[VIDEO] Pipeline en GPU ({time.time()-t0:.1f}s). VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB', flush=True)
    return pipeline

def generate_video(prompt: str, out_path: str, num_frames: int = 24, pipeline=None):
    import time
    import gc
    from diffusers.utils import export_to_video
    if pipeline is None:
        pipeline = get_video_pipeline()
    prompt = sanitize_video_prompt(prompt, max_len=150)
    num_frames = min(max(num_frames, 16), 64)
    print('[VIDEO] Generando frames en GPU (1-3 min)...', flush=True)
    t0 = time.time()
    out = pipeline(
        prompt,
        num_frames=num_frames,
        num_inference_steps=25,
    ).frames[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print(f'[VIDEO] Frames en {time.time()-t0:.1f}s. Exportando MP4...', flush=True)
    export_to_video(out, out_path, fps=8)
    del out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('[VIDEO] Video guardado.', flush=True)
    return out_path

In [8]:
# Celda 5: Subtitulos y composicion (MoviePy)
def make_srt_from_script(script: str, duration_sec: float) -> str:
    lines = [s.strip() for s in script.replace('.', '. ').split('.') if s.strip()]
    if not lines:
        lines = [script]
    n = len(lines)
    step = duration_sec / n if n else duration_sec
    srt = []
    for i, line in enumerate(lines):
        start = i * step
        end = (i + 1) * step
        srt.append(f'{i+1}\n{_ts(start)} --> {_ts(end)}\n{line}\n')
    return '\n'.join(srt)

def _ts(sec: float) -> str:
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int((sec % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def compose_video(video_path: str, audio_path: str, script: str, out_path: str, duration_audio: float):
    import time
    import gc
    import math
    from moviepy import VideoFileClip, AudioFileClip, concatenate_videoclips
    try:
        from moviepy.config import change_settings
    except ImportError:
        change_settings = lambda x: None
    print('[COMPOSE] Uniendo video + audio + subtitulos (resolucion reducida para ahorrar RAM)...', flush=True)
    t0 = time.time()
    try:
        import shutil
        imagick = shutil.which('convert') or '/usr/bin/convert'
        change_settings({'IMAGEMAGICK_BINARY': imagick})
    except Exception:
        pass
    video = VideoFileClip(video_path)
    audio = AudioFileClip(audio_path)
    target_dur = audio.duration
    if video.duration < target_dur:
        n = int(math.ceil(target_dur / video.duration))
        print(f'[COMPOSE] Video corto ({video.duration:.1f}s), repitiendo {n} veces para cubrir {target_dur:.1f}s', flush=True)
        subclip_fn = getattr(video, 'subclipped', getattr(video, 'subclip', None))
        clips = [subclip_fn(0, video.duration) for _ in range(n)]
        video = concatenate_videoclips(clips)
    end_sec = min(video.duration, target_dur)
    video = video.subclipped(0, end_sec) if hasattr(video, 'subclipped') else video.subclip(0, end_sec)
    video = video.with_audio(audio) if hasattr(video, 'with_audio') else video.set_audio(audio)
    target_h, target_w = 720, 405
    if video.w > video.h:
        video = video.resize((target_w, target_h))
    print(f'[COMPOSE] Clips cargados ({time.time()-t0:.1f}s). Creando subtitulos...', flush=True)
    dur = video.duration
    lines = [x.strip() for x in script.replace('?', '? ').split('.') if x.strip()]
    if not lines:
        lines = [script]
    srt_path = out_path.replace('.mp4', '_subs.srt')
    with open(srt_path, 'w', encoding='utf-8') as f:
        for i, line in enumerate(lines):
            step = dur / len(lines) if lines else dur
            start = i * step
            end = (i + 1) * step
            f.write(f'{i+1}\n')
            f.write(f'{_ts(start)} --> {_ts(end)}\n')
            f.write(line + '\n\n')
    tmp_path = out_path.replace('.mp4', '_tmp.mp4')
    print('[COMPOSE] Escribiendo MP4 (video+audio)...', flush=True)
    video.write_videofile(tmp_path, fps=24, codec='libx264', audio_codec='aac', threads=1)
    video.close()
    audio.close()
    del video
    import subprocess
    import os
    vf_sub = f"subtitles='{srt_path.replace(chr(39), chr(92)+chr(39))}':force_style=FontSize=22,PrimaryColour=&H00ffffff&"
    cmd = ['ffmpeg', '-y', '-i', tmp_path, '-vf', vf_sub, '-c:a', 'copy', out_path]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        print('[COMPOSE] Subtitulos quemados con ffmpeg.', flush=True)
    except Exception as e:
        print(f'[COMPOSE] ffmpeg subtitulos fallo ({e}), copiando sin subs.', flush=True)
        import shutil
        shutil.copy(tmp_path, out_path)
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    if os.path.exists(srt_path):
        try:
            os.remove(srt_path)
        except Exception:
            pass
    gc.collect()
    print(f'[COMPOSE] Listo en {time.time()-t0:.1f}s', flush=True)
    return out_path

In [9]:
# Celda 6: No precargamos modelos para ahorrar RAM. Se cargaran al generar (voz primero, luego video).
tts_model = None
print('Listo. Los modelos se descargaran/cargaran al pulsar Generar:', flush=True)
print('  1) Primero el modelo de VOZ (TTS) -> genera el audio.', flush=True)
print('  2) Se libera la voz para ahorrar RAM.', flush=True)
print('  3) Luego el modelo de VIDEO (Wan) -> genera las imagenes.', flush=True)
print('Revisa la consola de logs en la interfaz para ver cada paso.', flush=True)

Listo. Los modelos se descargaran/cargaran al pulsar Generar:
  1) Primero el modelo de VOZ (TTS) -> genera el audio.
  2) Se libera la voz para ahorrar RAM.
  3) Luego el modelo de VIDEO (Wan) -> genera las imagenes.
Revisa la consola de logs en la interfaz para ver cada paso.


In [10]:
# Celda 7: Pipeline completo. run_pipeline_gen hace yield tras cada paso para que la consola se actualice en vivo.
video_pipeline = None

_script_generator = None
def generate_script_from_topic(topic: str, max_len: int = 1500) -> str:
    '''Genera un guion de acertijo en espanol para 30-60 segundos (tema corto).'''
    global _script_generator
    topic = (topic or '').strip()[:200]
    if not topic:
        return ''
    try:
        if _script_generator is None:
            from transformers import pipeline
            _script_generator = pipeline('text2text-generation', model='google/flan-t5-small', device=-1)
        out = _script_generator(
            f'Write an engaging riddle script in Spanish for a 40 second video. Topic: {topic}. Include: short intro, riddle question with ?, then the answer. Use short sentences. About 100 words.',
            max_length=300,
            num_return_sequences=1,
        )
        text = (out[0].get('generated_text') or '').strip()
        return sanitize_script(text, max_len=max_len) if text else ''
    except Exception as e:
        print(f'[GUION] Error generando: {e}', flush=True)
        return ''

def expand_script_to_duration(script: str, target_words: int = 100) -> str:
    '''Expande un acertijo corto a un guion de ~30-60s (mas palabras para TTS).'''
    global _script_generator
    w = len((script or '').split())
    if w >= target_words:
        return script
    try:
        if _script_generator is None:
            from transformers import pipeline
            _script_generator = pipeline('text2text-generation', model='google/flan-t5-small', device=-1)
        out = _script_generator(
            f'Expand this Spanish riddle into a 40 second script. Add a short intro and conclusion. Keep the riddle and answer. Output about 100-120 words. Riddle: {script[:300]}',
            max_length=350,
            num_return_sequences=1,
        )
        text = (out[0].get('generated_text') or '').strip()
        return sanitize_script(text, max_len=1500) if text else script
    except Exception as e:
        print(f'[GUION] Error expandiendo: {e}', flush=True)
        return script

class TeeOut:
    def __init__(self, real, buf):
        self.real, self.buf = real, buf
    def write(self, s):
        self.real.write(s); self.buf.write(s)
    def flush(self):
        self.real.flush(); self.buf.flush()
    def close(self):
        self.flush()
    def isatty(self):
        return getattr(self.real, 'isatty', lambda: False)()

def run_pipeline(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    for path, msg, _ in run_pipeline_gen(texto_acertijo, prompt_video, speaker):
        pass
    return path, msg

def run_pipeline_gen(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    global tts_model, video_pipeline
    import time
    import gc
    import sys
    from io import StringIO
    log_buf = StringIO()
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout = TeeOut(old_stdout, log_buf)
    sys.stderr = TeeOut(old_stderr, log_buf)
    t0 = time.time()
    try:
        print('='*60, flush=True)
        print('[PIPELINE] INICIO. Consola activa: se actualiza en cada paso.', flush=True)
        print('='*60, flush=True)
        yield None, 'Iniciando pipeline...', log_buf.getvalue()
        if not texto_acertijo or not texto_acertijo.strip():
            yield None, 'Escribe el texto del acertijo.', log_buf.getvalue()
            return
        script = sanitize_script(texto_acertijo)
        if not script:
            yield None, 'Texto no valido.', log_buf.getvalue()
            return
        if len(script) < 100 and script.count('?') < 1:
            yield None, 'Paso 0/4 - Generando guion desde tu tema...', log_buf.getvalue()
            print('[PIPELINE] Paso 0/4 - Generando guion (tema corto detectado)...', flush=True)
            script = generate_script_from_topic(script)
            if not script:
                yield None, 'No se pudo generar el guion. Escribe el acertijo completo.', log_buf.getvalue()
                return
            print(f'[PIPELINE] Guion generado: {script[:80]}...', flush=True)
            yield None, 'Guion listo. Siguiente: voz.', log_buf.getvalue()
        if len(script.split()) < 70:
            yield None, 'Expandiendo guion para video 30-60s...', log_buf.getvalue()
            script = expand_script_to_duration(script, target_words=90)
        prompt_video = sanitize_video_prompt(prompt_video) if prompt_video else 'smooth fluid motion, cinematic, engaging, mysterious, vertical 9:16'
        base = OUTPUT_DIR / f'clip_{int(time.time())}'
        base.parent.mkdir(parents=True, exist_ok=True)
        audio_path = str(base) + '_audio.wav'
        video_path = str(base) + '_video.mp4'
        final_path = str(base) + '_final.mp4'
        duration_audio = 0.0
        try:
            yield None, 'Paso 1/4 - VOZ (cargando/generando audio)...', log_buf.getvalue()
            if tts_model is None:
                print('[PIPELINE] Paso 1/4 - VOZ: Cargando modelo de VOZ (TTS)...', flush=True)
                tts_model = get_tts_model()
            else:
                print('[PIPELINE] Paso 1/4 - VOZ: Generando audio...', flush=True)
            _, _, duration_audio = generate_voice(script, audio_path, speaker=speaker, model=tts_model)
            print(f'[PIPELINE] Paso 1/4 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            print('[PIPELINE] Liberando modelo de voz para liberar RAM...', flush=True)
            m = tts_model
            tts_model = None
            del m
            gc.collect()
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(1)
            print('[PIPELINE] RAM liberada. Cargando modelo de VIDEO en GPU.', flush=True)
            yield None, 'Paso 1/4 listo. Liberando voz.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en voz: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 2/4 - VIDEO (duracion segun audio)...', log_buf.getvalue()
            if not torch.cuda.is_available():
                yield None, 'Error: No hay GPU. Runtime > Change runtime type > GPU (T4).', log_buf.getvalue()
                return
            if video_pipeline is None:
                gc.collect()
                torch.cuda.empty_cache()
                print('[PIPELINE] Paso 2/4 - VIDEO: Cargando modelo ModelScope en GPU...', flush=True)
                video_pipeline = get_video_pipeline()
            else:
                print('[PIPELINE] Paso 2/4 - VIDEO: Generando frames...', flush=True)
            num_f = min(64, max(24, int(round(duration_audio * 8))))
            generate_video(prompt_video, video_path, num_frames=num_f, pipeline=video_pipeline)
            print(f'[PIPELINE] Paso 2/4 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            yield None, 'Paso 2/4 listo.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en video: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 3/4 - Composicion final (video + audio + subtitulos)...', log_buf.getvalue()
            print('[PIPELINE] Paso 3/4 - COMPOSICION...', flush=True)
            compose_video(video_path, audio_path, script, final_path, duration_audio)
            print(f'[PIPELINE] FIN. Tiempo total: {time.time()-t0:.1f}s', flush=True)
            yield final_path, f'Listo en {time.time()-t0:.1f}s', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error al componer: {e}', log_buf.getvalue()
    except Exception as e:
        import traceback
        traceback.print_exc()
        yield None, str(e), log_buf.getvalue()
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr


In [11]:
# Celda 8: Interfaz Gradio con pasos claros y ejemplos de prompts.
import gradio as gr

PROMPTS_VIDEO = [
    'smooth fluid motion, cinematic, engaging, mysterious atmosphere, vertical 9:16',
    'dark mood, riddle, enigma, soft lighting, cinematic, vertical 9:16',
    'dreamy, magical, fantasy, smooth motion, vertical 9:16',
    'mysterious forest, fog, cinematic, fluid camera, vertical 9:16',
    'abstract shapes, smooth animation, minimalist, vertical 9:16',
    'night sky, stars, contemplative, cinematic, vertical 9:16',
    'mysterious shadows, dramatic lighting, thriller mood, vertical 9:16',
    'soft bokeh, warm tones, intimate, cinematic, vertical 9:16',
]

EJEMPLOS_ACERTIJOS = [
    'Que cosa tiene dientes y no muerde? La sierra.',
    'Blanco es, la gallina lo pone, con aceite se frie y con pan se come. El huevo.',
    'Cuanto mas secas, mas mojadas estan. Las toallas.',
    'Tiene cabeza y no es persona, tiene corona y no es rey. La lechuga.',
    'Oro no es, plata no es. Abre las cortinas y lo veras. El platano.',
    'Sube y sube y nunca baja. La edad.',
]

def generar(texto, prompt_video, speaker):
    for path, msg, logs in run_pipeline_gen(texto, prompt_video, speaker):
        yield path, msg, logs

with gr.Blocks(title='Creador de Clips - Acertijos IG') as demo:
    gr.Markdown('# Creador de Clips - Acertijos IG')
    gr.Markdown(
        'Genera un video de 30-60 s con voz y subtitulos. **Pasos:** '
        '**0** Guion (si escribes tema corto) | **1** Voz (TTS) | **2** Video (IA) | **3** Composicion final. '
        'Tiempo estimado: 6-11 min.'
    )
    with gr.Row():
        with gr.Column(scale=1):
            txt_acertijo = gr.Textbox(
                label='Texto del acertijo o tema',
                placeholder='Pega un acertijo o escribe un tema corto (ej: la sierra).',
                lines=4,
            )
        with gr.Column(scale=1):
            dd_prompt = gr.Dropdown(
                label='Estilo de video (elegir ejemplo)',
                choices=PROMPTS_VIDEO,
                value=PROMPTS_VIDEO[0],
            )
            prompt_video = gr.Textbox(
                label='Descripcion del video (editable)',
                value=PROMPTS_VIDEO[0],
                lines=3,
            )
            dd_prompt.change(fn=lambda x: x, inputs=[dd_prompt], outputs=[prompt_video])
            speaker = gr.Dropdown(
                choices=['Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee'],
                value='Ryan',
                label='Voz',
            )
    gr.Markdown('**Ejemplos (clic en una fila para cargar acertijo + estilo + voz)**')
    gr.Examples(
        examples=[
            [EJEMPLOS_ACERTIJOS[0], PROMPTS_VIDEO[0], 'Ryan'],
            [EJEMPLOS_ACERTIJOS[1], PROMPTS_VIDEO[1], 'Vivian'],
            [EJEMPLOS_ACERTIJOS[2], PROMPTS_VIDEO[2], 'Serena'],
            ['la luna', PROMPTS_VIDEO[3], 'Ryan'],
            [EJEMPLOS_ACERTIJOS[4], PROMPTS_VIDEO[4], 'Aiden'],
        ],
        inputs=[txt_acertijo, prompt_video, speaker],
        label=None,
    )
    btn = gr.Button('Generar video', variant='primary')
    with gr.Row():
        out_video = gr.Video(label='Video resultado')
        with gr.Column():
            out_estado = gr.Textbox(label='Estado (paso actual)', interactive=False)
            out_logs = gr.Textbox(label='Consola (logs)', lines=16, max_lines=28, interactive=False)
    btn.click(
        fn=generar,
        inputs=[txt_acertijo, prompt_video, speaker],
        outputs=[out_video, out_estado, out_logs],
    )


In [12]:
# Celda 8b: Descargar modelos ANTES de lanzar la interfaz (ejecutar una vez; luego los pesos quedan en cache)
# Asi la primera generacion no se bloquea por la descarga. Opcional pero recomendado en Colab.
import gc
print('Pre-descarga: modelo de VOZ (Qwen TTS)...', flush=True)
_m = get_tts_model()
del _m
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Pre-descarga: modelo de VIDEO (ModelScope 1.7B)...', flush=True)
_p = get_video_pipeline()
del _p
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Listo. Modelos descargados en cache. Al pulsar Generar se cargaran desde disco.', flush=True)

Pre-descarga: modelo de VOZ (Qwen TTS)...
[TTS] Estamos en: descargando/cargando modelo de VOZ (Qwen3-TTS). Este modelo narra el acertijo.

********
********
 
[TTS] Dispositivo: cpu (GPU = menos RAM, mas rapido). Descargando pesos si es la primera vez...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[TTS] Modelo de voz listo en 23.5s
Pre-descarga: modelo de VIDEO (ModelScope 1.7B)...


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


...

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


[VIDEO] Pipeline en GPU (18.3s). VRAM: 3.7 GB


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `TextToVideoSDPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


Listo. Modelos descargados en cache. Al pulsar Generar se cargaran desde disco.


In [13]:
# Celda 9: Lanzar interfaz en Colab (enlace publico, 7 dias)
demo.launch(share=True, inline=False, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d84cde2061ee55776.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
